In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score

X, y = load_breast_cancer(return_X_y=True)

# Split 70/30 → then 70% of training again for blending
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_blend_train, X_meta, y_blend_train, y_meta = train_test_split(X_train, y_train, test_size=0.3)

models = [
    LogisticRegression(max_iter=200),
    SVC(probability=True),
    DecisionTreeClassifier()
]

# Train base models
for m in models:
    m.fit(X_blend_train, y_blend_train)

# Create meta-features
meta_features = np.column_stack([m.predict_proba(X_meta)[:, 1] for m in models])

meta_model = LogisticRegression()
meta_model.fit(meta_features, y_meta)

# Predict
test_meta = np.column_stack([m.predict_proba(X_test)[:, 1] for m in models])
pred = meta_model.predict(test_meta)

print("Accuracy:", accuracy_score(y_test, pred))